# Quality Operators: Data Assessment and Enrichment

This notebook demonstrates quality operators for assessing and improving data quality.

## Quality Operators Covered
1. **Language Detection** - Identify document language (176 languages)
2. **Readability Scoring** - Assess document readability
3. **PII/HAP Detection** - Detect sensitive content
4. **Deduplication** - Remove duplicate documents
5. **ML Enrichment** - Add quality metrics

## What You'll Learn
- How to assess document quality
- Configure quality operators
- Filter and clean data
- Enrich documents with metadata

## Prerequisites
- Sample documents available
- Ollama running (for PII/HAP detection)
- Virtual environment activated

## Setup and Imports

In [ ]:
import sys
from pathlib import Path
from pprint import pprint
import pandas as pd

# Add src to path if needed
import os
if 'PYTHONPATH' not in os.environ:
    src_path = Path.cwd().parent.parent / "src"
    sys.path.insert(0, str(src_path))

from docpipe.lib.docpipe_flow_manager import DocpipeFlowManager

print("✓ Imports loaded successfully")

## 1. Language Detection

Identify the language of documents using FastText (supports 176 languages):

In [ ]:
# Two invoice PDFs - one English, one Spanish - to demonstrate multilingual detection
language_flow = {
    "flow_name": "language-detection",
    "description": "Detect document language",
    "global_config": {
        "doc_column": "content",
        "disable_validation": False,
        "force_ingest": True
    },
    "flow": [
        {
            "name": "ingest",
            "type": "ingest_source",
            "config": {
                "provider": "filesystem",
                "connection_params": {"paths": ["../../tests/fixtures/language_docs"]},
                "include_filter": "pdf"

            }}
        },
        {
            "name": "extract_content",
            "type": "extract_operator",
            "depends_on": ["ingest"],
            "config": {
                "text_extraction": {
                    "provider": "docling_library"
                },
                "entity_extraction": {
                    "provider": "none"
                },
                "doc_column": "content"
            }
        },
        {
            "name": "language_detect",
            "type": "lang_detect",
            "depends_on": ["extract_content"],
            "config": {
                "provider": "fasttext",
                "doc_column": "content"
            }
        }
    ]
}

print("Detecting document languages...")
print("Input: English invoice + Spanish invoice (tests/fixtures/language_docs/)")
print("Provider: FastText (176 languages supported)")
print("Expected: 'en' for English invoice, 'es' for Spanish invoice")
print()

manager = DocpipeFlowManager(flow_def=language_flow)
manager.execute()

print("\n✓ Language detection completed!")
print("\nAdded Columns:")
print("  - language: Detected language code (e.g., 'en', 'es', 'fr')")
print("  - language_confidence: Confidence score (0-1)")
print("\nUse Cases:")
print("  - Filter documents by language")
print("  - Route to language-specific pipelines")
print("  - Validate expected language")

## 2. Readability Scoring

Calculate readability metrics (Flesch Reading Ease, Grade Level, etc.):

In [ ]:
readability_flow = {
    "flow_name": "readability-scoring",
    "description": "Calculate readability metrics",
    "global_config": {
        "doc_column": "content",
        "disable_validation": True,
        "force_ingest": True
    },
    "flow": [
        {
            "name": "ingest",
            "type": "ingest_source",
            "config": {
                "provider": "filesystem",
                "connection_params": {"paths": ["../../sample_documents"]},
                "include_filter": "txt"

            }}
        },
        {
            "name": "extract_content",
            "type": "extract_operator",
            "depends_on": ["ingest"],
            "config": {
                "text_extraction": {
                    "provider": "docling_library"
                },
                "entity_extraction": {
                    "provider": "none"
                },
                "doc_column": "content"
            }
        },
        {
            "name": "readability",
            "type": "readability",
            "depends_on": ["extract_content"],
            "config": {
                "doc_column": "content"
            }
        }
    ]
}

print("Calculating readability scores...")
print()

manager = DocpipeFlowManager(flow_def=readability_flow)
manager.execute()

print("\n✓ Readability scoring completed!")
print("\nReadability Metrics Added (Flesch, Gunning Fog, SMOG, Coleman-Liau, and more):")
print("  - flesch_reading_ease: 0-100 (higher = easier)")
print("  - flesch_kincaid_grade: US grade level")
print("  - gunning_fog, smog_index, coleman_liau_index, automated_readability_index")
print("  - dale_chall_readability_score, linsear_write_formula, spache_readability")
print("  - difficult_words, text_standard, mcalpine_eflaw, reading_time")
print("\nInterpretation (Flesch Reading Ease):")
print("  - 90-100: Very easy (5th grade)")
print("  - 60-70: Standard (8th-9th grade)")
print("  - 0-30: Very difficult (college graduate)")

## 3. Deduplication

Remove duplicate documents based on content similarity:

In [ ]:
dedup_flow = {
    "flow_name": "deduplication",
    "description": "Remove duplicate documents",
    "global_config": {
        "doc_column": "content",
        "disable_validation": False,
        "force_ingest": True
    },
    "flow": [
        {
            "name": "ingest",
            "type": "ingest_source",
            "config": {
                "provider": "filesystem",
                "connection_params": {"paths": ["../../tests/fixtures/customer_support_docs"]},
                "include_filter": "txt"

            }}
        },
        {
            "name": "extract_content",
            "type": "extract_operator",
            "depends_on": ["ingest"],
            "config": {
                "text_extraction": {
                    "provider": "docling_library"
                },
                "entity_extraction": {
                    "provider": "none"
                },
                "doc_column": "content"
            }
        },
        {
            "name": "dedup",
            "type": "ededup",
            "depends_on": ["extract_content"],
            "config": {
                "doc_column": "content",
                "doc_id_column": "doc_id_hash"
            }
        }
    ]
}

print("Removing duplicate documents...")
print("Note: Using customer support docs which contain duplicates")
print()

manager = DocpipeFlowManager(flow_def=dedup_flow)
manager.execute()

print("\n✓ Deduplication completed!")
print("\nHow it works:")
print("  - Computes content hash for each document")
print("  - Identifies exact duplicates")
print("  - Keeps first occurrence, removes duplicates")
print("\nUse Cases:")
print("  - Clean ingested data")
print("  - Reduce storage costs")
print("  - Improve search quality")

## 4. ML Enrichment

Add ML-based quality metrics (word counts, character ratios, etc.):

In [ ]:
ml_enrichment_flow = {
    "flow_name": "ml-enrichment",
    "description": "Add ML-based quality metrics",
    "global_config": {
        "doc_column": "content",
        "disable_validation": True,
        "force_ingest": True
    },
    "flow": [
        {
            "name": "ingest",
            "type": "ingest_source",
            "config": {
                "provider": "filesystem",
                "connection_params": {"paths": ["../../sample_documents"]},
                "include_filter": "txt"

            }}
        },
        {
            "name": "extract_content",
            "type": "extract_operator",
            "depends_on": ["ingest"],
            "config": {
                "text_extraction": {
                    "provider": "docling_library"
                },
                "entity_extraction": {
                    "provider": "none"
                },
                "doc_column": "content"
            }
        },
        {
            "name": "lang_detect",
            "type": "lang_detect",
            "depends_on": ["extract_content"],
            "config": {
                "doc_column": "content",
                "provider": "fasttext"
            }
        },
        {
            "name": "ml_enrichment",
            "type": "ml_enrichment",
            "depends_on": ["lang_detect"],
            "config": {
                "doc_column": "content",
                "lang_column": "lang_name",
                "language": "en"
            }
        }
    ]
}

print("Adding ML-based quality metrics...")
print()

manager = DocpipeFlowManager(flow_def=ml_enrichment_flow)
manager.execute()

print("\n✓ ML enrichment completed!")
print("\nMetrics Added (word counts, character ratios, duplication metrics, and more):")
print("  - Basic stats: num_words, num_chars, num_paragraphs, num_newlines")
print("  - Averages: avg_word_length, avg_paragraph_length")
print("  - Character ratios: alphanumeric, punctuation, control characters")
print("  - Duplication: paragraph duplicates, n-gram duplicates")
print("  - Special patterns: ellipsis, bullet points, tabs, hashes")
print("\nUse Cases:")
print("  - Filter low-quality documents")
print("  - Train quality classifiers")
print("  - Analyze corpus characteristics")

## 5. PII and HAP Detection

Detect Personally Identifiable Information (PII) and Hate/Abuse/Profanity (HAP):

**Note:** Requires Ollama running with a model

In [ ]:
pii_hap_flow = {
    "flow_name": "pii-hap-detection",
    "description": "Detect PII and HAP content",
    "global_config": {
        "doc_column": "content",
        "disable_validation": False,
        "force_ingest": True
    },
    "flow": [
        {
            "name": "ingest",
            "type": "ingest_source",
            "config": {
                "provider": "filesystem",
                "connection_params": {"paths": ["../../tests/fixtures/pii_hap_docs"]},
                "include_filter": "txt"

            }}
        },
        {
            "name": "extract_content",
            "type": "extract_operator",
            "depends_on": ["ingest"],
            "config": {
                "text_extraction": {
                    "provider": "docling_library"
                },
                "entity_extraction": {
                    "provider": "none"
                },
                "doc_column": "content"
            }
        },
        {
            "name": "pii_hap",
            "type": "pii_and_hap",
            "depends_on": ["extract_content"],
            "config": {
                "provider": "litellm",
                "doc_column": "content",
                "detect_pii": True,
                "detect_hap": True,
                "provider_config": {
                    "model_id": "openai/llama3.2",
                    "api_base": "http://localhost:11434/v1",
                    "api_key": "${OLLAMA_API_KEY}"
                },
                "pii_threshold": 0.5,
                "hap_threshold": 0.75,
                "expected_redactions": [
                    "pii",
                    "hap"
                ],
                "pii_list": [
                    "EmailAddress",
                    "PhoneNumber",
                    "SocialSecurityNumber",
                    "CreditCardNumber",
                    "BankAccountNumber"
                ],
                "redaction": True,
                "redaction_character": "*",
                "hap_redaction": True,
                "hap_redaction_character": "#",
                "display_pii": True,
                "batch_size": 4,
                "doc_column": "content"
            }
        }
    ]
}

print("Detecting PII and HAP content...")
print("Provider: Ollama (local LLM)")
print("Note: This may take a few moments")
print()

try:
    manager = DocpipeFlowManager(flow_def=pii_hap_flow)
    manager.execute()
    
    print("\n✓ PII/HAP detection completed!")
    print("\nPII Detection:")
    print("  - Names, emails, phone numbers")
    print("  - SSN, credit cards, addresses")
    print("  - Medical records, financial data")
    print("\nHAP Detection:")
    print("  - Hate speech")
    print("  - Abusive language")
    print("  - Profanity")
    print("\nAdded Columns:")
    print("  - pii_detected: Boolean flag")
    print("  - pii_types: List of detected PII types")
    print("  - hap_detected: Boolean flag")
    print("  - hap_categories: List of HAP categories")
except Exception as e:
    print(f"\n✗ PII/HAP detection failed: {e}")
    print("\nMake sure:")
    print("  1. Ollama is running: ollama serve")
    print("  2. Model is available: ollama pull llama3.2")

## 6. Complete Quality Pipeline

Combine multiple quality operators in a single pipeline:

In [ ]:
quality_pipeline = {
    "flow_name": "complete-quality-pipeline",
    "description": "Multi-stage quality assessment",
    "global_config": {
        "doc_column": "content",
        "disable_validation": True,
        "force_ingest": True
    },
    "flow": [
        {
            "name": "ingest",
            "type": "ingest_source",
            "config": {
                "provider": "filesystem",
                "connection_params": {"paths": ["../../sample_documents"]},
                "include_filter": "txt"

            }}
        },
        {
            "name": "extract_content",
            "type": "extract_operator",
            "depends_on": ["ingest"],
            "config": {
                "text_extraction": {
                    "provider": "docling_library"
                },
                "entity_extraction": {
                    "provider": "none"
                },
                "doc_column": "content"
            }
        },
        {
            "name": "language",
            "type": "lang_detect",
            "depends_on": ["extract_content"],
            "config": {
                "provider": "fasttext",
                "doc_column": "content"
            }
        },
        {
            "name": "readability",
            "type": "readability",
            "depends_on": ["language"],
            "config": {
                "doc_column": "content"
            }
        },
        {
            "name": "ml_enrichment",
            "type": "ml_enrichment",
            "depends_on": ["readability"],
            "config": {
                "doc_column": "content",
                "language": "en"
            }
        }
    ]
}

print("Running complete quality pipeline...")
print("Pipeline: Ingest → Extract → Language → Readability → ML Enrichment")
print()

manager = DocpipeFlowManager(flow_def=quality_pipeline)
manager.execute()

print("\n✓ Complete quality pipeline finished!")
print("\nDocuments now have:")
print("  - Language identification")
print("  - Readability scores (5 metrics)")
print("  - ML quality features (29 metrics)")
print("\nTotal: 35+ quality indicators per document")

## Best Practices

### 1. Quality Assessment Strategy
```python
# Start with fast operators
1. Language detection (fast)
2. Readability scoring (fast)
3. ML enrichment (fast)
4. Deduplication (medium)
5. PII/HAP detection (slow, LLM-based)
```

### 2. Filtering Low-Quality Documents
```python
# Use SQL filter after quality operators
{
    "type": "sql_filter",
    "config": {
        "query": "SELECT * WHERE flesch_reading_ease > 30 AND word_count > 100"
    }
}
```

### 3. Language-Specific Processing
```python
# Use branching operator for language-specific pipelines
{
    "type": "branching",
    "config": {
        "condition": "language == 'en'",
        "true_branch": "english_pipeline",
        "false_branch": "multilingual_pipeline"
    }
}
```

### 4. PII Handling
- Detect PII early in pipeline
- Use redaction operator to mask sensitive data
- Consider compliance requirements (GDPR, HIPAA)
- Log PII detection for audit trails

## Summary

You've learned:
1. ✓ Language detection with FastText (176 languages)
2. ✓ Readability scoring (Flesch, Gunning Fog, SMOG, Coleman-Liau, and more)
3. ✓ Document deduplication
4. ✓ ML-based quality enrichment (word counts, character ratios, duplication metrics, and more)
5. ✓ PII and HAP detection with LLMs
6. ✓ Building complete quality pipelines

## Next Steps

- **[06_rag_pipeline.ipynb](06_rag_pipeline.ipynb)** - Build end-to-end RAG workflow
- **[07_flow_authoring.ipynb](07_flow_authoring.ipynb)** - Programmatic flow creation
- **Combine with extraction** - Add quality checks after extraction
- **Filter and route** - Use quality metrics for intelligent routing

## Learn More

- [Language Detection Documentation](../../docs/operators/language_detection/language_detection_config.md)
- [Readability Documentation](../../docs/operators/readability/readability_config.md)
- [PII/HAP Documentation](../../docs/operators/pii_and_hap/pii_and_hap_config.md)
- [ML Enrichment Documentation](../../docs/operators/ml_enrichment/ml_enrichment_config.md)